In [1]:
"""
Code used to plot the graph showcasing results.
Each experiment appends to the existing .csv file, so we average values
across all experiments before plotting
"""

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os


FILE_MAPPING = {
    "ae_results.csv": "Autoencoder",
    "pca_results.csv": "PCA",
    "rsvd_results.csv": "RSVD",
    "grp_results.csv": "Gaussian RP",
    "srp_results.csv": "Sparse RP",
    "jl_results.csv": "JL (Hadamard)",
}


In [2]:
def load_all_data():
    """
    Loads all .csv files, adds a 'Method' column,
    and combines them into one DataFrame.
    """
    all_dataframes = []

    csv_files = glob.glob("../results/*.csv")

    if not csv_files:
        print("No csv files found.")
        return pd.DataFrame()

    print(f"Found {len(csv_files)} result files")

    for f in csv_files:
        filename = os.path.basename(f)

        method_name = FILE_MAPPING.get(filename)

        if method_name is None:
            print(f"Warning: Skipping file '{filename}' (not in FILE_MAPPING).")
            continue

        try:
            df = pd.read_csv(f)
            df["Method"] = method_name
            all_dataframes.append(df)
            print(f"  Loaded: {filename} (as '{method_name}')")
        except Exception as e:
            print(f"Error loading {filename}: {e}")

    if not all_dataframes:
        print("Error: No valid data was loaded.")
        return pd.DataFrame()

    # Combine all individual dataframes into one big one
    full_df = pd.concat(all_dataframes, ignore_index=True)

    full_df["dimension"] = pd.to_numeric(full_df["dimension"])
    full_df["avg_time_ms"] = pd.to_numeric(full_df["avg_time_ms"])
    full_df["accuracy(%)"] = pd.to_numeric(full_df["accuracy(%)"])

    return full_df

In [ ]:
def plot_accuracy_vs_dimension(df):
    plt.figure(figsize=(6, 4))
    sns.set_theme(style="whitegrid")

    plot = sns.lineplot(
        data=df,
        x="dimension",
        y="accuracy(%)",
        hue="Method",
        style="Method",
        markers=True,
        dashes=False,
        markersize=9,
        linewidth=2.5,
        estimator="mean",
        errorbar=("ci", 95),
    )

    plot.set_title("Accuracy vs. Embedding Dimension", fontsize=12, weight="bold")
    plot.set_xlabel("Dimension (Log Scale)", fontsize=11)
    plot.set_ylabel("Accuracy (%)", fontsize=11)

    plot.set_xscale("log", base=2)
    dims = sorted(df["dimension"].unique())
    plot.set_xticks(dims)
    plot.set_xticklabels(dims, fontsize=10)
    plot.invert_xaxis() # High dimension -> Low dimension

    plot.legend(loc='lower left', title=None, fontsize=9, framealpha=0.9, ncol=2)

    plt.grid(True, which="minor", ls=":", alpha=0.4)
    plot.tick_params(axis='both', labelsize=10)
    
    plt.tight_layout()
    plt.savefig("accuracy_vs_dimension.png", dpi=300, bbox_inches='tight')
    plt.close()
    

In [4]:
def plot_time_vs_dimension(df):
    plt.figure(figsize=(6, 4))
    sns.set_theme(style="whitegrid")
    
    df["dimension"] = pd.to_numeric(df["dimension"], errors="coerce")
    
    plot = sns.lineplot(
        data=df,
        x="dimension",
        y="avg_time_ms",
        hue="Method",
        style="Method",
        markers=True,
        dashes=False,
        markersize=9,
        linewidth=2.5,
        estimator="mean",
        errorbar=None,
    )

    plot.set_title("Matching Time vs. Dimension", fontsize=12, weight="bold")
    plot.set_xlabel("Dimension (Log Scale)", fontsize=11)
    plot.set_ylabel("Avg. Matching Time (ms)", fontsize=11)

    plot.set_xscale("log", base=2)
    dims = sorted(df["dimension"].unique())
    plot.set_xticks(dims)
    plot.set_xticklabels(dims, fontsize=10)
    plot.invert_xaxis()

    plot.legend(loc='best', title=None, fontsize=9, framealpha=0.9, ncol=2)

    plt.grid(True, which="minor", ls=":", alpha=0.4)
    plot.tick_params(axis='both', labelsize=10)

    plt.tight_layout()
    plt.savefig("time_vs_dimension.png", dpi=300, bbox_inches='tight')
    plt.close()


In [5]:

def get_best_markers_per_bin(df, x_col, y_col, method_col, bin_width=5):
    """
    Groups data by rounded x-intervals (bins) and keeps only the row 
    with the lowest y-value (EER) for each method in that bin.
    """
    temp_df = df.copy()
    temp_df['bin'] = (temp_df[x_col] / bin_width).round() * bin_width
    idx = temp_df.groupby([method_col, 'bin'])[y_col].idxmin()
    return temp_df.loc[idx]

def plot_tradeoff_curve(df):
    plt.figure(figsize=(7, 5))
    sns.set_theme(style="whitegrid")

    methods = sorted(df["Method"].unique())
    
    # Assign a specific color to each method
    unique_colors = sns.color_palette("deep", n_colors=len(methods))
    palette_dict = dict(zip(methods, unique_colors))
    
    # Assign a specific marker to each method
    available_markers = ['o', 'X', 's', 'P', 'D', '^', 'v']
    marker_dict = dict(zip(methods, available_markers[:len(methods)]))

    sns.lineplot(
        data=df,
        x="avg_time_ms",
        y="EER(%)",
        hue="Method",
        palette=palette_dict,  
        style="Method",        
        markers=False,         
        dashes=False,          
        linewidth=2.5,
        alpha=0.8,
        legend=True            
    )

    df_markers = get_best_markers_per_bin(df, "avg_time_ms", "EER(%)", "Method", bin_width=8)

    sns.scatterplot(
        data=df_markers,
        x="avg_time_ms",
        y="EER(%)",
        hue="Method",
        style="Method",
        palette=palette_dict,  
        markers=marker_dict,   
        s=100,
        zorder=10,
        edgecolor='white',
        linewidth=1.5,
        legend=False           
    )

    plt.title("EER (%) vs. Matching Time", fontsize=12, weight="bold")
    plt.xlabel("Matching Time (ms)", fontsize=11)
    plt.ylabel("EER (%)", fontsize=11)

    plt.legend(loc='best', fontsize=9, framealpha=0.9, ncol=2, title=None)

    plt.grid(True, which="minor", ls=":", alpha=0.4)
    plt.tight_layout()
    plt.savefig("eer_vs_time.png", dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Code for trade-off curves inspired by https://github.com/AndreisPurim/HEDimensionality

from matplotlib.ticker import ScalarFormatter

def plot_dimensionality_tradeoff(df):
    print("Generating 'dimensionality_tradeoff.png'...")
    
    method_to_plot = 'PCA'
    subset = df[df['Method'] == method_to_plot].copy()
    
    if subset.empty:
        print(f"No data for {method_to_plot}, skipping tradeoff plot.")
        return

    numeric_cols = ['dimension', 'avg_time_ms', 'EER(%)']
    subset = subset[numeric_cols].groupby('dimension').mean().reset_index()

    max_dim = subset['dimension'].max()
    baseline_row = subset[subset['dimension'] == max_dim]
    t_baseline = baseline_row['avg_time_ms'].values[0]
    
    # Sort Descending
    subset = subset.sort_values('dimension', ascending=False)

    x = subset['dimension'].values
    y_time_pct = (subset['avg_time_ms'].values / t_baseline) * 100
    y_eer = subset['EER(%)'].values

    fig, ax1 = plt.subplots(figsize=(6, 4))
    sns.set_theme(style="whitegrid")
    
    color_time = '#1f77b4'  # Tab:Blue
    color_eer = '#d62728'   # Tab:Red
    color_axis = 'black'
    
    ax1.set_xlabel("Embedding Dimension (n') - Log Scale", fontsize=11, color=color_axis)
    ax1.set_ylabel("Normalized Runtime: t'/t (%)", fontsize=11, color=color_axis)
    
    line1, = ax1.plot(x, y_time_pct, marker='s', color=color_time, linestyle='-', linewidth=2, markersize=8, label="Runtime")
    ax1.tick_params(axis='y', labelcolor=color_axis, labelsize=10)
    ax1.tick_params(axis='x', labelcolor=color_axis, labelsize=10)
    ax1.set_ylim(0, 105) 

    ax1.set_xscale('log')
    ax1.invert_xaxis()

    ax2 = ax1.twinx()  
    ax2.set_ylabel("Equal Error Rate (EER %)", fontsize=11, color=color_axis)
    ax2.grid(False)

    line2, = ax2.plot(x, y_eer, marker='o', color=color_eer, linestyle='--', linewidth=2, markersize=8, label="EER")
    ax2.tick_params(axis='y', labelcolor=color_axis, labelsize=10)

    # Custom Ticks
    ax1.set_xticks(x)
    ax1.get_xaxis().set_major_formatter(ScalarFormatter())
    ax1.minorticks_off()
    ax1.set_xticklabels([str(int(val)) for val in x]) 

    plt.title(f"Dimensionality Trade-Off ({method_to_plot})", fontsize=12, weight="bold", color=color_axis)
    
    # Grid settings
    ax1.grid(True, which="major", ls="-", alpha=0.5)
    
    lines = [line1, line2]
    labels = [line.get_label() for line in lines]
    

    legend = ax1.legend(lines, labels, loc='center left', bbox_to_anchor=(0.05, 0.5), fontsize=9, framealpha=0.9)
    
    plt.setp(legend.get_texts(), color='black')

    plt.tight_layout()
    plt.savefig("dimensionality_tradeoff.png", dpi=300, bbox_inches='tight')
    plt.close()

In [7]:
def main():
    df = load_all_data()

    if df.empty:
        return

    plot_accuracy_vs_dimension(df)
    plot_time_vs_dimension(df)
    plot_tradeoff_curve(df)
    plot_dimensionality_tradeoff(df)

if __name__ == "__main__":
    main()


Found 7 result files
  Loaded: rsvd_results.csv (as 'RSVD')
  Loaded: srp_results.csv (as 'Sparse RP')
  Loaded: ae_results.csv (as 'Autoencoder')
  Loaded: grp_results.csv (as 'Gaussian RP')
  Loaded: jl_results.csv (as 'JL (Hadamard)')
  Loaded: pca_results.csv (as 'PCA')
Generating 'dimensionality_tradeoff.png'...
